# Climate Data Exploratory Data Analysis: Ethiopia

## Project Overview
This notebook performs a comprehensive EDA on Ethiopia's climate dataset (2015-2026), focusing on temperature, precipitation, and wind patterns. The goal is to clean the data and identify key climate trends.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Data Loading & Date Parsing

In [ ]:
# Load the dataset
# Note: Assuming the file is in ../data/ethiopia.csv
try:
    df = pd.read_csv("../data/ethiopia.csv")
except FileNotFoundError:
    # Fallback to current directory for testing if needed
    df = pd.read_csv("ethiopia.csv")

# Add Country column
df["Country"] = "Ethiopia"

# Convert YEAR and DOY to datetime
df["DATE"] = pd.to_datetime(df["YEAR"] * 1000 + df["DOY"], format="%Y%j")

# Extract Month
df["Month"] = df["DATE"].dt.month

df.head()

### 2. Summary Statistics & Missing-Value Report

In [ ]:
# Replace NASA sentinel value -999 with NaN
df.replace(-999, np.nan, inplace=True)

# Check for duplicates
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows found: {duplicates}")
df.drop_duplicates(inplace=True)

# Summary Statistics
display(df.describe())

# Missing Values Report
missing_perc = (df.isna().sum() / len(df)) * 100
print("\nMissing Values Percentage per Column:")
print(missing_perc[missing_perc > 0])

# Flag columns with >5% nulls
high_nulls = missing_perc[missing_perc > 5]
if not high_nulls.empty:
    print(f"\nWARNING: Columns with >5% nulls: {high_nulls.index.tolist()}")

#### Interpretation:
- The mean temperature (T2M) provides a baseline for seasonal analysis.
- High variability in PRECTOTCORR suggests localized or intense rainfall events.
- Missing values are generally low; any column with >5% nulls might indicate sensor failure or out-of-range data for specific periods.

### 3. Outlier Detection & Cleaning

In [ ]:
cols_to_check = ['T2M', 'T2M_MAX', 'T2M_MIN', 'PRECTOTCORR', 'RH2M', 'WS2M', 'WS2M_MAX']
z_scores = np.abs(stats.zscore(df[cols_to_check].fillna(df[cols_to_check].mean())))
outliers_count = (z_scores > 3).sum()

print("Outlier counts (|Z| > 3):")
print(outliers_count)

# Handling outliers: We will cap them at the 1st/99th percentile to retain data while reducing noise.
for col in cols_to_check:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = np.clip(df[col], lower, upper)

# Remaining missing values: Forward-fill
df.fillna(method='ffill', inplace=True)

# Drop rows if more than 30% values are missing (sanity check)
df = df.dropna(thresh=int(df.shape[1] * 0.7))

# Export Cleaned Data
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/ethiopia_clean.csv', index=False)
print("Cleaned data exported to data/ethiopia_clean.csv")

### 4. Time Series Analysis

In [ ]:
monthly_avg = df.groupby(pd.Grouper(key='DATE', freq='ME'))['T2M'].mean()

plt.figure(figsize=(14, 6))
plt.plot(monthly_avg.index, monthly_avg.values, marker='o', linestyle='-', color='tab:red')

warmest = monthly_avg.idxmax()
coolest = monthly_avg.idxmin()

plt.annotate(f'Warmest: {monthly_avg.max():.1f}°C', xy=(warmest, monthly_avg.max()), 
             xytext=(warmest, monthly_avg.max()+1), arrowprops=dict(facecolor='black', shrink=0.05))
plt.annotate(f'Coolest: {monthly_avg.min():.1f}°C', xy=(coolest, monthly_avg.min()), 
             xytext=(coolest, monthly_avg.min()-1), arrowprops=dict(facecolor='black', shrink=0.05))

plt.title('Monthly Average Temperature (T2M) in Ethiopia (2015–2026)')
plt.ylabel('Temp (°C)')
plt.show()

In [ ]:
monthly_precip = df.groupby(pd.Grouper(key='DATE', freq='ME'))['PRECTOTCORR'].sum()

plt.figure(figsize=(14, 6))
plt.bar(monthly_precip.index, monthly_precip.values, width=20, color='tab:blue')
plt.title('Monthly Total Precipitation in Ethiopia')
plt.ylabel('Precipitation (mm)')
plt.show()

### 5. Correlation & Relationship Analysis

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(data=df, x='T2M', y='RH2M', ax=ax[0], alpha=0.5)
ax[0].set_title('T2M vs Relative Humidity (RH2M)')

sns.scatterplot(data=df, x='T2M_RANGE', y='WS2M', ax=ax[1], alpha=0.5)
ax[1].set_title('T2M_RANGE vs Wind Speed (WS2M)')
plt.show()

### 6. Distribution Analysis

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['PRECTOTCORR'], kde=True, log_scale=(False, True))
plt.title('Distribution of Daily Precipitation (Log Scale)')
plt.xlabel('Precipitation (mm)')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df.sample(500), x='T2M', y='RH2M', size='PRECTOTCORR', sizes=(20, 200), alpha=0.6)
plt.title('T2M vs RH2M (Bubble Size = Precipitation)')
plt.show()